# Statistical analysis of the data

This notebook consists of statistical checks of the inspected variables, related to the hypothesis set for this part of analysis: 

RQ: How do selected manifestations of political identity (political parties, party status, political orientation) shape parliamentary discourse in terms of sentiment?

- H1. Coalition parties will consistently exhibit higher neutral/positive sentiment, while opposition parties will display more negativity, though intensity may vary across terms based on political events.
- H2: Changes in political orientation will correlate with sentiment shifts, with right/traditionalist parties showing higher negativity.
- H3: Negative sentiment will dominate parliamentary debates, with minor occurrences of other sentiment categories. Political crises, elections, or leadership changes will correlate with sentiment shifts.

The sample illustrates the workflow. The full analysis is run on the complete dataset for final results.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats


In [2]:
df = pd.read_csv("../Results/Datasets/ParlaMint-SI_Family.tsv", sep="\t", encoding="utf-8")
df.head()

,ID,Text,Date,Term,Meeting,Subcorpus,Speaker_role,Speaker_MP,Speaker_minister,Speaker_party,...,Speaker_birth,Topic,Senti_3,Senti_6,Senti_n,Sents,Words,Tokens,Grouped_parties,Family
0,ParlaMint-SI_2014-01-27-SDZ6-Redna-21.u1,Spoštovane kolegice poslanke in kolegi poslanc...,2014-01-27,6. mandat,Redna,Reference,Chairperson,MP,notMinister,SD,...,1960,Mix,Neutral,neutral positive,3.21,25,504,570,ZLSD/SD,Socialist
1,ParlaMint-SI_2014-01-27-SDZ6-Redna-21.u2,"Hvala lepa za besedo. Zdaj Vlada predlaga, da ...",2014-01-27,6. mandat,Redna,Reference,Regular,MP,notMinister,SDS,...,1956,Macroeconomics,Negative,negative,0.00,25,419,481,SDS,Conservatives
2,ParlaMint-SI_2014-01-27-SDZ6-Redna-21.u3,Hvala tudi vam. O tej zahtevi ni razprave in o...,2014-01-27,6. mandat,Redna,Reference,Chairperson,MP,notMinister,SD,...,1960,Mix,Neutral,neutral positive,3.10,18,265,310,ZLSD/SD,Socialist
3,ParlaMint-SI_2014-01-27-SDZ6-Redna-21.u4,"Hvala lepa za besedo. To, kar se dogaja z nove...",2014-01-27,6. mandat,Redna,Reference,Regular,MP,notMinister,SDS,...,1956,Public Lands,Negative,negative,0.00,28,592,684,SDS,Conservatives
4,ParlaMint-SI_2014-01-27-SDZ6-Redna-21.u5,"Hvala. Naj vas opozorim, da tudi o tej zahtevi...",2014-01-27,6. mandat,Redna,Reference,Chairperson,MP,notMinister,SD,...,1960,Other,Neutral,neutral negative,1.84,28,405,471,ZLSD/SD,Socialist


### H1: Coalition parties more neutral/positive; opposition more negative

Given the non-normal distributions of sentiment checks, we will use Mann–Whitney U test to check several options: 

- a) is there a (statistically significant) difference between the two variables?
    - Null hypothesis (H₀): The distribution of sentiment scores is the same in coalition and opposition parties.

For this we will first run two-sided test (to compare if there is a difference between coalition and opposition in terms of sentiment).

- b) if the results are statistically significant, we will check the direction with the one-sided test
- c) Since some of speakers (and therefore parties) are very vocal as seen in the distribution checks, we run the test twice:
    - first run the test using raw sentiment scores, separated by coalition and opposition
    - for a second run we will compute average sentiment per party and repeat the test. 


In [16]:
# Two-sided Mann–Whitney U, raw sentiment scores
from scipy.stats import mannwhitneyu

coal_scores = df[df["Party_status"] == "Coalition"]["Senti_n"]
oppo_scores = df[df["Party_status"] == "Opposition"]["Senti_n"]

u_stat, p_two = mannwhitneyu(coal_scores, oppo_scores, alternative="two-sided")
print("Mann–Whitney U statistic:", u_stat)
print("Two-sided p-value:", p_two)

n1, n2 = len(oppo_scores), len(coal_scores)
effect_size_r = 1 - (2*u_stat) / (n1*n2)
print("Effect size (r):", effect_size_r)
print("Coalition median:", coal_scores.median())
print("Opposition median", oppo_scores.median())



Mann–Whitney U statistic: 1645543.0
Two-sided p-value: 3.046903445819607e-70
Effect size (r): -0.36230653041473726
Coalition median: 3.0
Opposition median 1.18


Given the statistically significant results in the two-sided test, we then test for direction with a one-sided test

In [20]:
#One-sided Mann-Whitney U test, raw sentiment scores

u_stat, p_one = mannwhitneyu(coal_scores, oppo_scores, alternative='greater')

print("One-sided p-value (coalition > opposition):", p_one)

# Rank-biserial effect size
n1, n2 = len(coal_scores), len(oppo_scores)
r = 1 - (2*u_stat)/(n1*n2)
print("Effect size (rank-biserial r):", r)

One-sided p-value (coalition > opposition): 1.5234517229098035e-70
Effect size (rank-biserial r): -0.36230653041473726
